> ## SOLUTION / ANSWER KEY &mdash; Lab 3.3
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-03-attention-by-hand.ipynb`](../lab-03-attention-by-hand.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 3.3 &mdash; Self-Attention by Hand

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Implement a numerically-stable softmax
- Compute scaled dot-product attention: softmax(Q.Kt / sqrt(d)) . V
- See a query 'attend' to the matching key

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Self-attention (Q/K/V)](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = "/tmp/biaa-lab-03-03"
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
**Attention** lets each token look at every other token and pull in what's relevant. For queries
**Q**, keys **K** and values **V**:
`attention(Q, K, V) = softmax( Q . Kt / sqrt(d) ) . V`.
The `softmax` turns similarity scores into weights that sum to 1; the scaling by `sqrt(d)` keeps them
stable. This is the exact maths a real model runs &mdash; here we compute it by hand so the mechanism
is unambiguous (Lab 3.9 pulls the same weights out of a real model).

## Build it
Implement `softmax` and `attention`.

In [ ]:
import numpy as np
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d)
    weights = softmax(scores, axis=-1)
    return weights @ V

## Run it for real
Run attention on a tiny example where the answer is obvious.

In [ ]:
try:
    import numpy as np
    Q = np.array([[10.0, 0.0], [0.0, 10.0]])   # two strong queries
    K = np.array([[1.0, 0.0], [0.0, 1.0]])     # two orthogonal keys
    V = np.array([[1.0, 0.0], [0.0, 1.0]])     # values to fetch
    print("attention output:\n", np.round(attention(Q, K, V), 3))
    print("softmax([2,1,0]) =", np.round(softmax(np.array([2.0, 1.0, 0.0])), 3), "(sums to 1)")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## What to notice
- The output is (almost) the identity: query 1 pulls value 1, query 2 pulls value 2 &mdash; each **strong query attended to its matching key**.
- Softmax weights are a **probability distribution** &mdash; they sum to 1, so attention is a weighted average of the values.
- Drop the `/ sqrt(d)` scaling and, for large `d`, the softmax saturates (one weight ~1, the rest ~0) &mdash; that is the vanishing-gradient problem the scaling fixes.

## Your turn (open task &mdash; no grader)
Make the two queries **similar** instead of orthogonal (e.g. `[[5,5],[5,5]]`) &mdash; what
happens to the attention weights and the output? Then grow `d` and add the scaling back and forth to
feel its effect. A "good" answer: you can predict the output shape and roughly where the weight mass
lands before you run it.

---
### Key takeaway
That is the whole mechanism. A transformer stacks many of these attention steps -- and that is what 'Attention Is All You Need' meant.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>